In [ ]:
'''Imports.'''

%load_ext autoreload
%autoreload 2

from utils.config import TEMP_SOLD_DIR
from utils.io import read_df_from_path, read_multiple_df
from stats.cleaning import (
    FilterConfig,
    drop_empty_records, remove_trailing_name_text, 
    keep_records_containing, remove_records_containing, 
    remove_trailing_name_text, conv_dates, set_date_as_index,
    replace_null_shipping_with_avg
    )
from stats.visualisation import show_price_hist
from stats.calculations import fit_to_lognormal, fit_to_normal
from stats.outliers import filter_by_percentile, filter_by_price, filter_by_standard_deviation
from stats.generate_pdf import pdf_normal, pdf_lognormal

In [ ]:
'''Load and view raw data.
SPECIFY PRODUCT NAME HERE'''

PRODUCT_NAME = "iphone 15"

csv_path = TEMP_SOLD_DIR / PRODUCT_NAME
raw_df = read_multiple_df(csv_path) #reads all df for a product at once

raw_df.head()

In [ ]:
'''Filter data'''

filter_conf = FilterConfig(
    required_txts = [PRODUCT_NAME],
    banned_txts = [
    # --- 1. Model Variants ---
    "pro", "promax", "pro max", "plus", "mini", 
    
    # --- 2. Wrong Storage Capacities ---
    # Catching variations with and without spaces/letters
    "256gb", "256 gb", "256g", "256",
    "512gb", "512 gb", "512g", "512",
    "1tb", "1 tb", "1 t", "1024gb",
    "64gb", "64 gb", "64g", "64", # Older models just in case
    
    # --- 3. Accessories & "Not a Phone" Scams ---
    "case", "cover", "protector", "screen guard", 
    "box only", "empty box", "replacement box", 
    "dummy", "fake", "clone", "cable", "charger", "magsafe",
    ]
)


filtered_df = (
    raw_df
    .pipe(drop_empty_records)
    .pipe(remove_trailing_name_text)
    .pipe(keep_records_containing, filter_conf)
    .pipe(remove_records_containing, filter_conf)
    .pipe(conv_dates)
    .pipe(set_date_as_index)
    .pipe(replace_null_shipping_with_avg)
)

# TODO: Get data from last N days
filtered_df.sort_values(by="price")

In [ ]:
'''Looking at specific records.'''

filtered_df.head().columns

In [ ]:
'''Visualising data.'''

show_price_hist(filtered_df)

In [ ]:
#data seems bimodal, multiple items present
#fixed by fitering out iphone pros as well, now fits lognormal distribution

In [ ]:
'''Clean data using std'''

cleaned_df = filter_by_standard_deviation(filtered_df, n=2)

show_price_hist(cleaned_df)

In [ ]:
simple_stats = cleaned_df["price"].describe()
normal_params = fit_to_normal(cleaned_df)
lognormal_params = fit_to_lognormal(cleaned_df)

print(simple_stats)

max = simple_stats["max"]
min = simple_stats["min"]
mean = simple_stats["mean"]
std = normal_params["std"]

lines_to_include = {
    "mean": mean,
    "mean+std": mean+std,
    "mean-std": mean-std,
    "q1": simple_stats["25%"],
    "q3": simple_stats["75%"],
}

normal_pdf = pdf_normal(normal_params, max=max, min=min)
lognormal_pdf = pdf_lognormal(lognormal_params, max=max, min=min)

show_price_hist(cleaned_df, dist_values=lognormal_pdf, lines=lines_to_include, bin_size=20)
